In [1]:
### KNN ###

import numpy as np
from experimental_design import ParameterSpace, NA
from classification_icsa import get_param_space, get_param_configs
import warnings
import multiprocessing as mp
import logging

# NOTE: this assumes ordinal variables have values in increasing order

parameter_domain = dict(
    n_neighbors=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30],
    weights=['uniform', 'distance'],
    p=[NA(), 1, 2],
    metric=['minkowski', 'braycurtis', 'canberra', 'chebyshev', 'cosine']
)

parameter_type = dict(
    n_neighbors='o',
    weights='c',
    p='r',
    metric='c'
)

parameter_default = dict(
    n_neighbors=5,
    weights='uniform',
    p=2,
    metric='minkowski'
)

In [2]:
ps = get_param_space('knn', parameter_domain, parameter_type, parameter_default)

Saved analysis/classification/knn/parameter_space.txt
This can be imported at https://srd.sba-research.org/tools/cagen/ to generate a covering array. Export to .csv and save the file in the same directory as parameter_space.txt. Remember to include any constraints or conditionality in the Constraints field of CAGen.
[System]
Name: parameter_space

[Parameter]
n_neighbors(enum): 0,1,2,3,4,5,6,7,8,9,10,11,12,13
weights(enum): 0,1
p(enum): 0,1,2
metric(enum): 0,1,2,3,4

[Constraint]



In [3]:
# REMEMBER: constraints:
# metric = "0" => p != "0"
# metric != "0" => p = "0"

get_param_configs(t=4, ps=ps, use_log_uniform=[])

In [ ]:
import re
from sklearn.neighbors import KNeighborsClassifier
from classification_icsa import run_classifier_configs
import warnings

def value_error_correction(classifier, value_error):
    import re
    match = re.search(r"n_samples_fit = (\d+)", str(value_error))
    n_samples_fit = int(match.group(1))
    classifier.n_neighbors = min(classifier.n_neighbors, n_samples_fit)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    run_classifier_configs(KNeighborsClassifier, value_error_correction, 'knn')

  0%|          | 0/235 [00:00<?, ?it/s]

In [1]:
### SGDClassifier ###

import numpy as np
from experimental_design import ParameterSpace, NA
from classification_icsa import get_param_space, get_param_configs
import warnings
# import ray
import multiprocessing as mp
import logging
# ray.init(logging_level=logging.ERROR, ignore_reinit_error=True)

# NOTE: this assumes ordinal variables have values in increasing order

parameter_domain = dict(
    loss = ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron'],
    penalty = ['l2', 'l1', 'elasticnet', None],
    alpha = [NA(), (1e-6, 1e-5), (1e-5, 1e-4), 1e-4, (1e-4, 1e-3), (1e-3, 1e-2), (1e-2, 1e-1)],
    l1_ratio = [NA(), 0.0, (0.0, 0.2), 0.15, (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0), 1.0],
    class_weight = [None, 'balanced'],
    average = [False, True],
    learning_rate = ['constant', 'optimal', 'invscaling', 'adaptive'],
    eta0 = [NA(), (1e-4, 1e-3), (1e-3, 1e-2), (1e-2, 1e-1), (1e-1, 1), (1, 10)],
    power_t = [NA(), (0, 0.2), (0.2, 0.4), 0.5, (0.4, 0.6), (0.6, 0.8), (0.8, 1)]
)

parameter_type = dict(
    loss = 'c',
    penalty = 'c',
    alpha = 'r',
    l1_ratio = 'r',
    class_weight = 'c',
    average = 'c',
    learning_rate = 'c',
    eta0 = 'r',
    power_t = 'r'
)

parameter_default = dict(
    loss = 'hinge',
    penalty = 'l2',
    alpha = 1e-4,
    l1_ratio = 0.15,
    class_weight = None,
    average = False,
    learning_rate = 'optimal',
    eta0 = 1e-4,
    power_t = 0.5
)

In [2]:
ps = get_param_space('sgdclassifier', parameter_domain, parameter_type, parameter_default)

Saved analysis/classification/sgdclassifier/parameter_space.txt
This can be imported at https://srd.sba-research.org/tools/cagen/ to generate a covering array. Export to .csv and save the file in the same directory as parameter_space.txt. Remember to include any constraints or conditionality in the Constraints field of CAGen.
[System]
Name: parameter_space

[Parameter]
loss(enum): 0,1,2,3,4
penalty(enum): 0,1,2,3
alpha(enum): 0,1,2,3,4,5,6
l1_ratio(enum): 0,1,2,3,4,5,6,7,8
class_weight(enum): 0,1
average(enum): 0,1
learning_rate(enum): 0,1,2,3
eta0(enum): 0,1,2,3,4,5
power_t(enum): 0,1,2,3,4,5,6

[Constraint]



In [3]:
def print_constraints_sgd(parameter_domain):
    penalty_None_index = str(parameter_domain['penalty'].index(None))
    learning_rate_optimal_index = str(parameter_domain['learning_rate'].index('optimal'))
    alpha_constr = f'alpha = "0" => (penalty = "{penalty_None_index}" && learning_rate != "{learning_rate_optimal_index}")\n' +\
        f'alpha != "0" => (penalty != "{penalty_None_index}" || learning_rate = "{learning_rate_optimal_index}")'
    
    penalty_elasticnet_index = str(parameter_domain['penalty'].index('elasticnet'))
    l1_ratio_constr = f'l1_ratio = "0" => penalty != "{penalty_elasticnet_index}"\n' +\
        f'l1_ratio != "0" => penalty = "{penalty_elasticnet_index}"'
    
    eta0_constr = f'eta0 = "0" => learning_rate = "{learning_rate_optimal_index}"\n' +\
        f'eta0 != "0" => learning_rate != "{learning_rate_optimal_index}"'
    
    learning_rate_invscaling_index = str(parameter_domain['learning_rate'].index('invscaling'))
    power_t_constr = f'power_t = "0" => learning_rate != "{learning_rate_invscaling_index}"\n' +\
        f'power_t != "0" => learning_rate = "{learning_rate_invscaling_index}"'
    print(
        alpha_constr + '\n' +
        l1_ratio_constr + '\n' +
        eta0_constr + '\n' +
        power_t_constr
    )

print_constraints_sgd(parameter_domain)

alpha = "0" => (penalty = "3" && learning_rate != "1")
alpha != "0" => (penalty != "3" || learning_rate = "1")
l1_ratio = "0" => penalty != "2"
l1_ratio != "0" => penalty = "2"
eta0 = "0" => learning_rate = "1"
eta0 != "0" => learning_rate != "1"
power_t = "0" => learning_rate != "2"
power_t != "0" => learning_rate = "2"


In [4]:
# REMEMBER: constraints as input to CAgen

get_param_configs(t=3, ps=ps, use_log_uniform=['alpha', 'eta0'])

In [1]:
import re
from sklearn.linear_model import SGDClassifier
from classification_icsa import run_classifier_configs
import warnings

def value_error_correction(classifier, value_error):
    if classifier.alpha is None:
        classifier.alpha = 0.0001
    if classifier.l1_ratio is None:
        classifier.l1_ratio = 0.15
    if classifier.power_t is None:
        classifier.power_t = 0.5
    if classifier.eta0 is None:
        classifier.eta0 = 0.0
    pass


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    run_classifier_configs(SGDClassifier, value_error_correction, 'sgdclassifier',
                           multicore_mode='configs')

100%|██████████| 235/235 [35:01<00:00,  8.94s/it] 
